# Example 7: 使用用户子程序 (UMAT) 运行 N 个 Job

演示 ABQflow 的用户子程序支持：`SubroutineSpec` + `abaqus make` 编译 + `user=` 参数透传。

本例使用一个线弹性 UMAT（改编自
`reference/abaqus_skills/abaqus_subroutine_skills/reference/material/umat_elastic.md`），
通过 `*User Material` 关键字接管原本由 Abaqus 内置 `*Elastic` 处理的本构关系。
由于是纯弹性模型，UMAT 给出的结果理论上应该和内置 `*Elastic` 材料（见 Example 1）
完全一致 —— 这也是验证 UMAT 编译/传参是否正确的最简单方法。

**环境要求**：本地需要安装 Intel Fortran 或 gfortran 才能编译 `.for` 源文件
（`abaqus make`），并确保 Abaqus 版本与编译器兼容。

In [9]:
import os

from ABQflow import (
	BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec, SubroutineSpec,
)

os.environ.pop("NoDefaultCurrentDirectoryInExePath", None)

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/2026/abaqus.bat'
CWD = os.getcwd()
OUTPUT_DIR = os.path.join(CWD, "examples/07_SubroutineJob/output")
UMAT_SOURCE = os.path.join(CWD, "examples/07_SubroutineJob/subroutine/umat_elastic.for")

## 构造 N 个 Job

INP 模板 `planar_stress_umat_template.inp` 和 Example 1 用的模板有两处区别：

1. 把 `*Elastic` 换成了 `*User Material, constants=2`（`PROPS(1)=E, PROPS(2)=NU`，
   和原模板的 `{{youngs_modulus}}, 0.3` 参数完全对应）；
2. 加了一段模型级的 `*Section Controls, hourglass=ENHANCED`。网格里的 CPS4R 是
   减缩积分单元，Abaqus 没法从 `*User Material` 推出默认的沙漏刚度（否则会报
   *"elements have been defined with zero hour glass stiffness"* 并直接终止）；
   ENHANCED 会用 UMAT 第一次返回的 Jacobian 来算，因此不需要再引入一个参数。

每个 Job 通过 `subroutine=` 字段指向同一个 UMAT 源文件 ——
`SubroutineCompileStrategy` 会在每个 Job 自己的 `output_dir` 里各自编译一次
（带哈希缓存，源码不变时后续重跑会跳过编译）。

In [10]:
YOUNGS_MODULUS_LIST = [190000, 200000, 210000]

specs = [
	JobSpec(
		job_name = f"umat_job_{i:02d}",
		workflow = "modular",
		preparation = PreparationSpec(
			kind = "inp_based",
			source_path = "./examples/cae_file/planar_stress_umat_template.inp",
			params = {
				"youngs_modulus": e,
				"load_magnitude": 2000,
			}
		),
		subroutine = SubroutineSpec(
			source_path = UMAT_SOURCE,
			language = "fortran",
			solver = "standard",
		),
		post_extraction = [
			HookSpec(
				script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
				tasks = [
					{"result_name": "max_stress_mises",},
					{"result_name": "max_displacement",},
				]
			)
		]
	)
	for i, e in enumerate(YOUNGS_MODULUS_LIST, start=1)
]

len(specs)

3

In [11]:
processor = BatchAbaqusProcessor(
	batch_data = specs,
	base_output_dir = OUTPUT_DIR,
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

## Dry-run 预览：确认编译 + `user=` 命令行

`dry_run('plan')` 不产生任何副作用，纯粹根据 spec 拼出会执行的命令行，
可以在真正提交前确认 `abaqus make` 和 `user=` 参数是否符合预期。

In [12]:
plans = processor.dry_run("plan")

for p in plans:
	print(f"--- {p.job_name} ---")
	for cmd in p.commands:
		print(f"  [{cmd.stage}] {' '.join(cmd.cmd)}")

--- umat_job_01 ---
  [compile] C:/Applications/SIMULIA/Commands/2026/abaqus.bat make library=c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/07_SubroutineJob/subroutine/umat_elastic.for
  [solver] C:/Applications/SIMULIA/Commands/2026/abaqus.bat job=umat_job_01 input=c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/07_SubroutineJob/output\umat_job_01\umat_job_01.inp cpus=4 user=c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/07_SubroutineJob/subroutine/umat_elastic.for interactive
  [hook:./examples/extraction_scripts/get_max_stress_mises.py] C:/Applications/SIMULIA/Commands/2026/abaqus.bat python ./examples/extraction_scripts/get_max_stress_mises.py --odb_path c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/07_SubroutineJob/output\umat_job_01\umat_job_01.odb --job_name umat_job_01 --tasks_json <generated-at-runtime>
--- umat_job_02 ---
  [compile] C:/Applications/SIMULIA/Commands/2026/abaqus.bat make library=c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/07_SubroutineJob/subroutine/umat_elasti

## 正式运行

每个 Job 依次执行：编译 UMAT → 前处理生成 INP → 求解（`user=umat_elastic.for`）→ 后处理提取。

In [13]:
outcomes = processor.run_batch(num_parallel_jobs=3)

[09/07/26 15:38:03] WARNING  Existing job dirs: ['umat_job_01', 'umat_job_02', 'umat_job_03']

Output()

In [14]:
outcomes

[JobOutcome(job_name='umat_job_01', status='COMPLETED', results={'max_stress_mises': 4525.22900390625, 'max_displacement': 4.405528545379639}, error=None, diagnostics=None, output_dir='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples/07_SubroutineJob/output\\umat_job_01', phases=[{'phase': 'compile', 'status': 'COMPILED', 'started_at': 1788766683.7723362, 'ended_at': 1788766686.1391075, 'duration_s': 2.3667712211608887, 'error': None}, {'phase': 'preparation', 'status': 'PREPARATION_SUCCESS', 'started_at': 1788766686.1391075, 'ended_at': 1788766686.1426232, 'duration_s': 0.0035157203674316406, 'error': None}, {'phase': 'simulation', 'status': 'SIMULATION_SUCCESS', 'started_at': 1788766686.1426232, 'ended_at': 1788766732.5054157, 'duration_s': 46.36279249191284, 'error': None}, {'phase': 'post_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1788766732.5054157, 'ended_at': 1788766734.5844638, 'duration_s': 2.0790481567382812, 'error': None}], duration_s=50.81363272666931),
 J

## 检查结果 + 编译状态

`JobOutcome.phases` 里应该能看到 `compile` 阶段，用来确认子程序确实被编译了
（而不是意外走了默认弹性材料）。

In [15]:
for oc in outcomes:
	print(oc.job_name, oc.status)
	if oc.status == "COMPLETED":
		print("  max_stress_mises =", oc.results["max_stress_mises"])
		print("  max_displacement =", oc.results["max_displacement"])
	else:
		print("  error:", oc.error)

for oc in outcomes:
	compile_phase = next((p for p in (oc.phases or []) if p["phase"] == "compile"), None)
	print(oc.job_name, "compile phase:", compile_phase)

umat_job_01 COMPLETED
  max_stress_mises = 4525.22900390625
  max_displacement = 4.405528545379639
umat_job_02 COMPLETED
  max_stress_mises = 4525.22900390625
  max_displacement = 4.1852521896362305
umat_job_03 COMPLETED
  max_stress_mises = 4525.22900390625
  max_displacement = 3.985954523086548
umat_job_01 compile phase: {'phase': 'compile', 'status': 'COMPILED', 'started_at': 1788766683.7723362, 'ended_at': 1788766686.1391075, 'duration_s': 2.3667712211608887, 'error': None}
umat_job_02 compile phase: {'phase': 'compile', 'status': 'COMPILED', 'started_at': 1788766683.7893264, 'ended_at': 1788766686.24443, 'duration_s': 2.455103635787964, 'error': None}
umat_job_03 compile phase: {'phase': 'compile', 'status': 'COMPILED', 'started_at': 1788766683.7893264, 'ended_at': 1788766686.2454343, 'duration_s': 2.4561078548431396, 'error': None}


## 验证正确性

Example 1 里同样 `youngs_modulus=210000` 的内置 `*Elastic` 材料给出
`max_stress_mises ≈ 4525.26`。下面把 UMAT 版本（`umat_job_03`，E=210000）的结果
和这个参考值做对比 —— 因为 UMAT 实现的就是同一个各向同性线弹性本构，
两者应该非常接近（数值误差在浮点精度范围内）。

In [16]:
REFERENCE_MAX_STRESS_MISES = 4525.26025390625  # Example 1, youngs_modulus=210000

results = {oc.job_name: oc.results for oc in outcomes}

umat_result = results["umat_job_03"]
diff = abs(umat_result["max_stress_mises"] - REFERENCE_MAX_STRESS_MISES)
print(f"UMAT max_stress_mises = {umat_result['max_stress_mises']}")
print(f"reference (built-in *Elastic) = {REFERENCE_MAX_STRESS_MISES}")
print(f"abs diff = {diff}")
assert diff < 1.0, "UMAT elastic result should match the built-in *Elastic material closely"

UMAT max_stress_mises = 4525.22900390625
reference (built-in *Elastic) = 4525.26025390625
abs diff = 0.03125
